### Explore Raw Schema
Read `test_scans.json`, preview 10 records, print the schema, and get total record/column counts.

### 1. Spark Session
Local Spark session tuned for the 74 GB JSON file.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test_scans_explore")
    .master("local[6]")
    .config("spark.driver.memory", "12g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.caseSensitive", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/19 12:53:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 2. Read JSON (Cached Schema)
Infers the schema once and caches it, since inference on 74 GB can take 10-20 minutes. Later runs reuse the cache and skip inference.

In [4]:
import json, os
from pathlib import Path
from pyspark.sql.types import StructType

# OUTPUT_ROOT = this repo's root, one level above this notebook (pyspark_files/..).
# Raw input JSON and generated artifacts (schema cache, CSVs) all live under this repo's data/ folder.
OUTPUT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
JSON_PATH = os.path.join(OUTPUT_ROOT, "data", "raw_data", "test_scans.json")
SCHEMA_CACHE_PATH = os.path.join(OUTPUT_ROOT, "data", "cache", "test_scans_schema.json")

if Path(SCHEMA_CACHE_PATH).exists():
    schema = StructType.fromJson(json.loads(Path(SCHEMA_CACHE_PATH).read_text()))
    df = spark.read.schema(schema).json(JSON_PATH)
    print(f"loaded cached schema from {SCHEMA_CACHE_PATH} ({len(df.columns)} top-level fields)")
else:
    print("no schema cache found; inferring from full file (this can take 10-20 min)...")
    df = (
        spark.read
        .option("dropFieldIfAllNull", "true")
        .json(JSON_PATH)
    )
    Path(SCHEMA_CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)
    Path(SCHEMA_CACHE_PATH).write_text(df.schema.json())
    print(f"wrote schema cache to {SCHEMA_CACHE_PATH} ({len(df.columns)} top-level fields)")


no schema cache found; inferring from full file (this can take 10-20 min)...


wrote schema cache to /Users/vaddegurudevaraju/Desktop/CyberSecurity/Restrctured_repository/data/cache/test_scans_schema.json (186 top-level fields)


### 3. Preview 10 Records

In [ ]:
df.show(10, truncate=80)


### 4. Print Schema

In [ ]:
df.printSchema()


### 5. Record & Column Counts
Total rows and columns, shown as one summary row.

In [ ]:
total_records = df.count()
total_columns = len(df.columns)

summary_df = spark.createDataFrame(
    [(total_records, total_columns)],
    ["total_records", "total_columns"],
)
summary_df.show()
